In [4]:
import os
import shutil

ruta_carpeta = '../data/raw/dogs-vs-cats/train/'
for filename in os.listdir(ruta_carpeta):
    if filename.startswith('cat'):
            destino = os.path.join('../data/raw/', 'cat')
            shutil.move(os.path.join(ruta_carpeta, filename), os.path.join(destino, filename))
            print(f'Movido {filename} a la carpeta cat')
    elif filename.startswith('dog'):
            destino = os.path.join('../data/raw/', 'dog')
            shutil.move(os.path.join(ruta_carpeta, filename), os.path.join(destino, filename))
            print(f'Movido {filename} a la carpeta dog')



FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/dogs-vs-cats/train/'

In [5]:
import random

ruta_base = '../data/raw/'

clases = ['cat', 'dog']
ratio_train_test = 0.8 #80% para entrenamiento, 20% para prueba

#Semilla para reproducibilidad
random.seed(42)

ruta_dataset = os.path.join(ruta_base, 'dataset')

for clase in clases:
    #armo la ruta de la clase
    ruta_clase = os.path.join(ruta_base, clase)
    #listo las imágenes de la clase
    imagenes = [archivo for archivo in os.listdir(ruta_clase)]
    #las mezclo aleatoriamente
    random.shuffle(imagenes)

    #calculo el número de imágenes para entrenamiento y prueba
    num_train = int(len(imagenes) * ratio_train_test)
    #dejo la lista de imágenes de entrenamiento y prueba
    imagenes_train = imagenes[:num_train]
    imagenes_test = imagenes[num_train:]

    #mover a train
    for imagen in imagenes_train:
        ruta_origen = os.path.join(ruta_clase, imagen)
        ruta_destino = os.path.join(ruta_dataset, 'train', clase, imagen)
        shutil.move(ruta_origen, ruta_destino)
        print(f'Movido {imagen} a la carpeta train/{clase}')

    #mover a test
    for imagen in imagenes_test:
        ruta_origen = os.path.join(ruta_clase, imagen)
        ruta_destino = os.path.join(ruta_dataset, 'test', imagen)
        shutil.move(ruta_origen, ruta_destino)
        print(f'Movido {imagen} a la carpeta test/{clase}')

In [ ]:

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Definir los generadores de datos para entrenamiento y prueba
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Crear los generadores de datos a partir de las carpetas
train_generator = train_datagen.flow_from_directory(
    '../data/raw/dataset/train',
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    '../data/raw/dataset/test',
    target_size=(150, 150),
    batch_size=32,
    shuffle=False
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [ ]:
from tensorflow.keras.applications import VGG16

#Cargar modelo preentrenado VGG16 sin la capa superior (clasificación)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))

#Congelar las capas del modelo base para que no se entrenen durante el ajuste fino
base_model.trainable = False

#Crear el modelo secuencial y agregar las capas del modelo base
model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # Salida binaria (gato o perro)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_generator,
    epochs=3
)

In [ ]:
import numpy as np

predicciones = model.predict(test_generator)

predicciones

In [ ]:
clases_predichas = (predicciones > 0.5).astype(int).ravel()  # Convertir a 0 o 1 y a un array unidimensional

etiquetas_mapa = {v: k for k, v in train_generator.class_indices.items()}
resultados = [etiquetas_mapa[clase] for clase in clases_predichas]

for archivo, resultado in zip(test_generator.filenames[:10], resultados[:10]):
    print(f'Archivo: {archivo}, Predicción: {resultado}')

In [ ]:
loss, accuracy = model.evaluate(test_generator)

print(f'Pérdida en el conjunto de prueba: {loss}')
print(f'Precisión en el conjunto de prueba: {accuracy}')

In [ ]:
model.save('../models/cats_vs_dogs_model.keras')